# Install and load dependencies

In [ ]:
# ! pip install -q evaluate bitsandbytes
# ! pip install -qU datasets wandb

In [ ]:
import os
import torch
import evaluate
import wandb
import tempfile
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader
from kaggle_secrets import UserSecretsClient

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    get_scheduler,
    AutoModelForSequenceClassification,
)

# Config

In [ ]:
checkpoint = "bert-base-uncased"
num_train_epochs = 2
train_bs = 16
gacc_steps = 2                      # gradient accumulation steps
lr_scheduler_type = "linear"
lr = 5e-5
max_grad_norm = 1.0                 # gradient clipping threshold
use_8bit_adam = True
logging_steps = 10
eval_on_start = True

os.environ["WANDB_LOG_MODEL"] = "end"
os.environ["WANDB_WATCH"] = "false"
wandb_project = "HFLLM-transformer-finetuning"
wandb_group   = "[torch_trainer]bert-mrpc-analysis"
wandb_run_name = "test_run_2"

# W&B Login

In [ ]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

# Load dataset
- [MRCP Dataset](https://aclanthology.org/)

In [ ]:
raw_datasets = load_dataset("glue", "mrpc")

In [ ]:
print("====== overall split and scehma")
display(raw_datasets)
print("====== a single record")
display(raw_datasets["train"][0])
print("====== what does each label value mean?")
display(raw_datasets["train"].features)
print("====== saved dataset file structure (Apache Arrows File)")
os.listdir("/root/.cache/huggingface/datasets/")

# Preprocess the dataset

In [ ]:
def tokenize_function(example, tokenizer):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
    
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns = ['sentence1', 'sentence2', 'idx'],
    fn_kwargs = {'tokenizer': tokenizer}
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Test the prep pipeline

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items()}
print("========= lengths before collating/padding")
print("input_ids: ", [len(x) for x in samples["input_ids"]])
print("attention_mask: ", [len(x) for x in samples["attention_mask"]])
print("========= lengths after collating/padding")
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

## Some additional steps
- These are handled by the Hugging Face `Trainer` automatically

In [ ]:
# rename "label" to "labels"
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
# set format to torch.Tensor
tokenized_datasets.set_format("torch")

# Train

## Components

### Dataloaders

In [ ]:
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=train_bs, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=train_bs, collate_fn=data_collator
)

batch = next(iter(train_dataloader))
print({k: v.shape for k, v in batch.items()})

### Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

### Optimizer

- The default optimizer of the HF Trainer is `AdamW`
- `Adam` incorporates weight decay into the moving averages of gradients (L2 regularization),
    - which can weaken the regularization effect.
- `AdamW` decouples this, applying weight decay directly to the parameters,
    - leading to more effective, independent regularization.
- `bnb` can be slightly slower in some cases due to extra quant/dequant overhead.

In [ ]:
if use_8bit_adam:
    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=lr)
        print("Using bitsandbytes AdamW8bit optimizer.")
    except Exception as e:
        print(f"Could not use bitsandbytes 8-bit optimizer (falling back to torch AdamW). Reason: {e}")
        from torch.optim import AdamW
        optimizer = AdamW(model.parameters(), lr=lr)
else:
    from torch.optim import AdamW
    optimizer = AdamW(model.parameters(), lr=lr)

### Learning rate scheduler

In [ ]:
num_update_steps_per_epoch = (len(train_dataloader) + gacc_steps - 1) // gacc_steps
num_training_steps = num_train_epochs * num_update_steps_per_epoch
eval_steps = max(1, num_update_steps_per_epoch // 4)

lr_scheduler = get_scheduler(
    lr_scheduler_type,
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
print("Train batches per epoch:", len(train_dataloader))
print("Gradient accumulation steps:", gacc_steps)
print("Optimizer updates per epoch:", num_update_steps_per_epoch)
print("Total optimizer updates (num_training_steps):", num_training_steps)

### Scaler and AMP

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
bf16_supported = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if bf16_supported else torch.float16
use_amp = (device.type == "cuda")
scaler_enabled = (use_amp and amp_dtype == torch.float16) # not needed for bf16 since it's much more stable than fp16
scaler = torch.amp.GradScaler(device.type, enabled=scaler_enabled)
print(f"AMP enabled: {use_amp} | amp_dtype: {amp_dtype} | scaler enabled: {scaler.is_enabled()}")

## Monitoring and experiment tracking

In [ ]:
wandb.init(
    project=wandb_project,
    group=wandb_group,
    name=wandb_run_name,
    config={
        "checkpoint": checkpoint,
        "num_train_epochs": num_train_epochs,
        "train_bs": train_bs,
        "gradient_accumulation_steps": gacc_steps,
        "effective_batch_size": train_bs * gacc_steps,
        "lr": lr,
        "lr_scheduler_type": lr_scheduler_type,
        "max_grad_norm": max_grad_norm,
        "use_8bit_adam": use_8bit_adam,
        "amp_dtype": str(amp_dtype),
        "bf16_supported": bool(bf16_supported),
    }
)

## Evaluation

In [ ]:
def run_eval(model, eval_dataloader, device, amp_dtype, use_amp):
    model.eval()
    metric = evaluate.load("glue", "mrpc")
    total_loss = 0.0
    n_batches = 0

    for batch in tqdm(eval_dataloader,desc=f"Evaluation"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                outputs = model(**batch)
                loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

        preds = torch.argmax(outputs.logits, dim=-1)
        metric.add_batch(predictions=preds, references=batch["labels"])

    m = metric.compute()
    return {
        "eval_loss": total_loss / max(1, n_batches),
        "eval_accuracy": m["accuracy"],
        "eval_f1": m["f1"],
    }

## Main loop

In [ ]:
global_step = 0               # counts OPTIMIZER UPDATES (not micro-batches)
running_loss = 0.0            # accumulates true (un-divided) losses for logging
running_mb = 0                # number of micro-batches accumulated into running_loss

best_eval_loss = float("inf")
best_state_dict = None

model.to(device);

train_progress_bar = tqdm(total=num_training_steps, desc="Optimizer Updates")

if eval_on_start:
    eval_logs = run_eval(model, eval_dataloader, device, amp_dtype, use_amp)
    if eval_logs["eval_loss"] < best_eval_loss:
        best_eval_loss = eval_logs["eval_loss"]
        best_state_dict = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
    wandb.log(eval_logs, step=global_step)
    print("Eval on start:", eval_logs)


for epoch in range(num_train_epochs):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        # autocast forward
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            outputs = model(**batch)
            loss = outputs.loss
            loss = loss / gacc_steps

        true_loss = outputs.loss.detach().float().item()
        running_loss += true_loss
        running_mb += 1

        # backward with scaler
        scaler.scale(loss).backward()

        # Do an optimizer step every gacc_steps (or at end)
        is_accum_step = ((step + 1) % gacc_steps == 0) or ((step + 1) == len(train_dataloader))
        if is_accum_step:
            # Gradient clipping (must unscale first when using scaler)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            if global_step % logging_steps == 0:
                avg_train_loss = running_loss / max(1, running_mb)
                wandb.log(
                    {
                        "train_loss": avg_train_loss,
                        "learning_rate": lr_scheduler.get_last_lr()[0],
                        "epoch": epoch + (step + 1) / len(train_dataloader),
                    },
                    step=global_step
                )
                running_loss = 0.0
                running_mb = 0
            if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                eval_logs = run_eval(model, eval_dataloader, device, amp_dtype, use_amp)
                wandb.log(eval_logs, step=global_step)
                print(f"Step {global_step} eval:", eval_logs)
                
                if eval_logs["eval_loss"] < best_eval_loss:
                    best_eval_loss = eval_logs["eval_loss"]
                    best_state_dict = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }

            train_progress_bar.update(1)
print("Done.")

## Push the best checkpoint to W&B

In [ ]:
assert best_state_dict is not None, "Best model state was never captured. Did eval run?"
model.load_state_dict(best_state_dict)

with tempfile.TemporaryDirectory() as tmpdir:
    model.save_pretrained(tmpdir, safe_serialization=True)
    tokenizer.save_pretrained(tmpdir)

    artifact = wandb.Artifact(
        name=f"{wandb_run_name}-best",
        type="model",
        metadata={
            "best_eval_loss": float(best_eval_loss),
            "best_step": int(wandb.summary.get("best_step", global_step)),
            "checkpoint": checkpoint,
            "train_bs": train_bs,
            "gacc_steps": gacc_steps,
            "effective_batch_size": train_bs * gacc_steps,
            "lr": lr,
            "lr_scheduler_type": lr_scheduler_type,
            "amp_dtype": str(amp_dtype),
            "use_8bit_adam": bool(use_8bit_adam),
        },
    )
    artifact.add_dir(tmpdir)
    wandb.log_artifact(artifact)
wandb.finish()